# 四个接触距离的有限对照

固定同一轨迹、帧、边界与对象名单，只改变距离：0.7、0.8、0.9、1.0 nm。
序列排除≤2；伙伴阈值1、5、10。中心链与全部浓相链分别比较。
均值先帧内链平均再对帧等权；误差线为时间块SEM，分布阴影为块SD，不是置信区间。
概率曲线是每个残基至少参与一次有效分子内接触的概率；高概率饱和影响分辨率，但不自动证明阈值错误。
只读本研究产物，不读取原始轨迹或更改正式参数。


In [ ]:
from importlib import import_module
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulations.yaml").is_file())
sys.path.insert(0, str(ROOT))
from common import inputs
from common import results as files

CUT_OFFS = [0.7, 0.8, 0.9, 1.0]
SELECTED = None
OUTPUT = ROOT / "cutoff_comparison/output"
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SIMULATIONS = inputs.select_simulations(ROOT / "simulations.yaml", SELECTED)
tables = import_module('cutoff_comparison.dependencies.tables')
summary, profiles, distributions, diagnostics, SOURCES = tables.collect(
    OUTPUT, SIMULATIONS, CUT_OFFS
)
ORDER = [s["id"] for s in SIMULATIONS]
COLORS = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(ORDER)}
OBJECTS = ("dense_chains", "central_chain")
LABELS = {"dense_chains": "All dense chains", "central_chain": "Central dense chain"}
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
    }
)


def export(fig, name):
    for key, record in SOURCES.items():
        path = files.completed(OUTPUT / key)
        if files.read_json(path / "metadata.json") != record:
            raise ValueError("Results changed; rerun notebook")
    for ext in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    files.write_json(FIGURES / f"{name}.sources.json", SOURCES)


summary.to_csv(FIGURES / "all_summary.csv", index=False)
diagnostics.to_csv(FIGURES / "profile_diagnostics.csv", index=False)
display(diagnostics)


In [ ]:
metrics = {
    "intra_contacts": "Internal contact pairs / chain",
    "inter_contacts": "External residue contacts / chain",
    "partner_count": "Partner chains (threshold ≥1)",
    "participation": "Residues with an internal contact",
}
fig, axes = plt.subplots(2, 4, figsize=(15, 7), constrained_layout=True)
for row, obj in enumerate(OBJECTS):
    for ax, (metric, label) in zip(axes[row], metrics.items()):
        for name in ORDER:
            data = summary[
                summary.object.eq(obj) & summary.simulation_id.eq(name) & summary.threshold.eq(1)
            ].sort_values("cutoff_nm")
            sem = data[metric + "_sem_blocks"].to_numpy()
            ax.errorbar(
                data.cutoff_nm,
                data[metric + "_mean_frames"],
                yerr=sem if np.isfinite(sem).any() else None,
                color=COLORS[name],
                marker="o",
                capsize=3,
                label=name,
            )
        ax.set(xlabel="Contact distance (nm)", ylabel=label, xticks=CUT_OFFS, title=LABELS[obj])
        if metric == "participation":
            ax.set_ylim(0, 1.02)
axes[0, 0].legend()
fig.suptitle("Only distance varies: frame-equal mean ± block SEM", fontsize=11)
export(fig, "distance_vs_metrics")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), constrained_layout=True)
for row, obj in enumerate(OBJECTS):
    for ax, threshold in zip(axes[row], (1, 5, 10)):
        for name in ORDER:
            data = summary[
                summary.object.eq(obj)
                & summary.simulation_id.eq(name)
                & summary.threshold.eq(threshold)
            ].sort_values("cutoff_nm")
            sem = data.partner_count_sem_blocks.to_numpy()
            ax.errorbar(
                data.cutoff_nm,
                data.partner_count_mean_frames,
                yerr=sem if np.isfinite(sem).any() else None,
                marker="o",
                capsize=3,
                color=COLORS[name],
                label=name,
            )
        ax.set(
            xlabel="Contact distance (nm)",
            ylabel="Partner chains",
            xticks=CUT_OFFS,
            title=f"{LABELS[obj]} · ≥{threshold} contacts",
        )
axes[0, 0].legend()
export(fig, "distance_vs_partner_thresholds")
plt.show()


In [ ]:
for obj in OBJECTS:
    fig, axes = plt.subplots(
        1, 4, figsize=(15, 3.5), sharex=True, sharey=True, constrained_layout=True
    )
    for ax, cutoff in zip(axes, CUT_OFFS):
        for name in ORDER:
            data = profiles[
                profiles.object.eq(obj)
                & profiles.simulation_id.eq(name)
                & profiles.cutoff_nm.eq(cutoff)
            ].sort_values("residue_index")
            x, y, sd = (
                data.residue_index.to_numpy(),
                data["mean"].to_numpy(),
                data.std_blocks.to_numpy(),
            )
            ax.plot(x, y, color=COLORS[name], label=name, lw=1.2)
            ax.fill_between(
                x, np.maximum(0, y - sd), np.minimum(1, y + sd), color=COLORS[name], alpha=0.12
            )
        ax.set(xlabel="Residue index (0-based)", title=f"{cutoff:.1f} nm", ylim=(0, 1.02))
    axes[0].set_ylabel("P(at least one internal contact)")
    axes[0].legend()
    fig.suptitle(f"{LABELS[obj]}: mean ± block SD", fontsize=11)
    export(fig, f"residue_probability_{obj}")
    plt.show()


In [ ]:
PARTNER_MAX = 30
for obj in OBJECTS:
    fig, axes = plt.subplots(
        3, 4, figsize=(14, 8), sharex=True, sharey=True, constrained_layout=True
    )
    for row, threshold in enumerate((1, 5, 10)):
        for ax, cutoff in zip(axes[row], CUT_OFFS):
            for name in ORDER:
                data = distributions[
                    distributions.object.eq(obj)
                    & distributions.simulation_id.eq(name)
                    & distributions.cutoff_nm.eq(cutoff)
                    & distributions.threshold.eq(threshold)
                ].sort_values("partner_count")
                x, y, sd = (
                    data.partner_count.to_numpy(),
                    data["mean"].to_numpy(),
                    data.std_blocks.to_numpy(),
                )
                ax.plot(x, y, color=COLORS[name], label=name, lw=1.2)
                ax.fill_between(
                    x, np.maximum(0, y - sd), np.minimum(1, y + sd), color=COLORS[name], alpha=0.12
                )
            ax.set(
                xlim=(0, PARTNER_MAX),
                xticks=np.arange(0, PARTNER_MAX + 1, 5),
                ylim=(0, None),
                title=f"{cutoff:.1f} nm · ≥{threshold} contacts",
                xlabel="Partner chains",
                ylabel="Mean frame fraction",
            )
    axes[0, 0].legend()
    fig.suptitle(LABELS[obj], fontsize=11)
    export(fig, f"partner_distributions_{obj}")
    plt.show()
